# Reinforcement Learning with Verifiable Rewards (RLVR)

In Session 15 we changed a model's weights with GRPO, rewarding it for verifiably correct answers. This session zooms into the other half of that loop — the **verifier** — and the data pipeline built around it. No GPU required: we run the RLVR sampling-and-verification loop against an API model, so the focus stays on the part that makes or breaks reinforcement learning on language models: the reward signal itself.

The RLVR loop looks like this:

```text
prompt -> sample N completions -> verify each against a deterministic checker
       -> assign rewards -> keep verified-correct samples as preference data
       -> policy update -> repeat
```

Unlike RLHF, there is no learned reward model and no human labeler in the loop. The reward comes from a *deterministic program* — a math answer checker, a unit-test runner — that either passes a completion or doesn't. That makes the signal cheap, objective, and reproducible. It also makes it a target: any policy trained against a verifier will find and exploit its blind spots, so we will also build reward-hacking detection and an audit trail that records every verifier decision.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how RLVR differs from RLHF, and what makes a reward "verifiable."
- Implement verifiable reward functions for math (exact-answer matching) and code (unit-test execution).
- Run the sample-and-verify loop and interpret group-level accuracy.
- Detect reward-hacking signatures and maintain a verifier audit trail.
- Construct chosen/rejected preference pairs ready for DPO-style training — or reward signals ready for GRPO.

## Table of Contents

- **Breakout Room #1: The Verifiable Reward Loop**
  - Task 1: Environment Setup
  - Task 2: Problems and Answer Extraction
  - Task 3: A Math Reward Function
  - Question #1 and Question #2
  - Task 4: Sample and Verify
- **Breakout Room #2: Reward Hacking, Code Verification, and Preference Data**
  - Task 5: Reward-Hacking Detection and the Audit Trail
  - Question #3
  - Task 6: A Code Verifier
  - Question #4
  - Task 7: Build Preference Pairs
  - Activity #1
- **Conclusion: What We Built, Start to Finish**
- **What Looks Different in Production**

---
# Breakout Room #1
## The Verifiable Reward Loop

We build the core RLVR machinery: a small set of math problems with known answers, a deterministic answer checker, a reward function, and the sample-and-verify loop that turns them into training signal.

## Task 1: Environment Setup

From the `16_RLVR` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

You will need an [OpenAI API key](https://platform.openai.com/api-keys). Enter it below — it is kept in memory for this session only.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

### The Policy

In RL terms, the model we sample from is the **policy**. We deliberately use a small model (`gpt-4.1-nano`) at temperature 1.0: a policy that is *sometimes wrong* is exactly what we want, because the contrast between verified-correct and verified-incorrect samples is where the training signal lives. A policy that never fails produces no gradient — and no preference pairs.

In [2]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-4.1-nano"  # small on purpose: we *want* some wrong answers


def simple_complete(prompt: str, system: str = "", temperature: float = 1.0) -> str:
    """One completion from the policy model."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


print(simple_complete("Reply with exactly: policy online"))

policy online


## Task 2: Problems and Answer Extraction

RLVR only works in domains where correctness can be *checked by a program*. Math word problems with a single numeric answer are the canonical example (this is why GSM8K shows up in every RLVR paper — and in Session 15).

Two conventions make checking reliable:

1. Each problem carries a **ground-truth answer** as a string.
2. The prompt instructs the policy to put its final answer in `\boxed{}` — the same convention GSM8K-style training uses — so extraction is a regex, not a judgment call. If no box is found, we fall back to the last number in the text.

In [3]:
import re
from dataclasses import dataclass


@dataclass
class Problem:
    question: str
    answer: str  # ground truth


problems = [
    Problem("What is 12 * 13?", "156"),
    Problem("If a train covers 60 km in 45 minutes, what is its speed in km/h?", "80"),
    Problem("What is the sum of the first 10 positive integers?", "55"),
    Problem(
        "A store discounts a $250 jacket by 20%, then adds 10% sales tax on the discounted price. "
        "What is the final price in dollars?",
        "220",
    ),
    Problem("How many positive divisors does 360 have?", "24"),
]


def extract_number(text: str) -> str:
    r"""Pull the final answer out of a completion: \boxed{...} first, last number as fallback."""
    boxed = re.search(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed.group(1).strip()
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else ""


assert extract_number(r"Step 1: 12*13 = 156. The answer is \boxed{156}.") == "156"
assert extract_number("So the speed is 80 km/h") == "80"
assert extract_number("no numbers here") == ""
print("extraction OK")

extraction OK


## Task 3: A Math Reward Function

The reward function is where verification becomes training signal:

- **+1.0** if the extracted answer matches the ground truth,
- **−0.1** otherwise.

Note the asymmetry: full credit for success, only a *small* penalty for failure. We also normalize numerically (`"80"` and `"80.0"` should match) — a verifier that fails on formatting technicalities punishes correct reasoning, which is the fastest way to teach a policy the wrong lesson.

In [4]:
class MathRewardFunction:
    """Verifiable reward for math problems: exact answer match against ground truth."""

    correct_reward = 1.0
    incorrect_penalty = -0.1

    def compute(self, response: str, ground_truth: str) -> float:
        prediction = extract_number(response)
        if self._normalize(prediction) == self._normalize(ground_truth):
            return self.correct_reward
        return self.incorrect_penalty

    @staticmethod
    def _normalize(value: str):
        """Compare numerically when possible, so '80' == '80.0'."""
        try:
            return float(value)
        except ValueError:
            return value.strip()


reward_fn = MathRewardFunction()

assert reward_fn.compute(r"The speed is \boxed{80}", "80.0") == 1.0
assert reward_fn.compute(r"The speed is \boxed{81}", "80") == -0.1
assert reward_fn.compute("I cannot solve this.", "80") == -0.1
print("reward function OK")

reward function OK


#### ❓ Question #1

In your own words: what makes a reward "verifiable," and how does RLVR differ from RLHF's learned reward model? Give one task where a verifiable reward exists and one where it fundamentally cannot (and explain why).

##### Answer:

* *A reward is "verifiable" when a deterministic program, not a model or a human, can check whether the output is correct against a known ground truth. This is what RLVR relies on: math answers matched against a solution, code scored by unit tests. There is no intermediary that can drift, be gamed as a proxy, or need retraining.*
    * **Verifiable task**: *solving a math word problem with a single numeric answer, since correctness is a mechanical check against ground truth.*

* *RLHF instead trains a separate reward model on human preference data, then uses that model's score as the training signal. That reward model is itself an approximation of human judgment, so it can be miscalibrated, exploited, or go stale as the policy's outputs shift away from what it was trained on.*
    * **Non-verifiable task**: *judging whether a response is empathetic or well-written, since there is no ground truth to check against. Only human judgment (or a learned proxy for it) can score that, which is exactly why RLHF exists for tasks like this.*


#### ❓ Question #2

The reward is asymmetric: +1.0 for a correct answer but only −0.1 for an incorrect one. Suppose we used −1.0 instead. What behavior might a policy learn during early training, when most of its attempts fail? (Hint: think about a model that discovers it can hedge, refuse, or produce no parseable answer at all.)

##### Answer:

*In early training, correctness is rare, so most rollouts fail. With a −1.0 penalty, the expected reward of actually attempting the problem is close to the same as any degenerate shortcut, since every wrong attempt gets punished as harshly as possible regardless of effort. The −0.1 penalty in the notebook keeps failed attempts cheap enough that the policy still has room to explore different reasoning paths on the way to finding correct ones.*

*Under −1.0, the policy has no differential signal telling it that "genuine attempt, wrong answer" is better than "hedge, refuse, or produce nothing parseable." Both land at the same harsh penalty, but hedging or refusing is cheaper to produce (shorter, lower-variance completions with less risk of compounding mistakes). So the policy can learn to collapse toward these low-effort, risk-averse outputs rather than continuing to explore toward the correct answer, since exploring offers no better expected reward at this stage but costs more.*

*This mirrors a known RLHF failure mode: overly harsh penalties for being wrong push models toward over-refusal and hedging instead of genuine effort, because refusing is the safer, lower-variance action once failure is punished as severely as possible.*

## Task 4: Sample and Verify

Now the heart of RLVR: for each problem, sample a **group** of completions at temperature 1.0 and verify every one.

Sampling groups (rather than one completion per prompt) is not incidental — it is the same structure GRPO consumed in Session 15, where each completion's advantage was computed *relative to its group's average reward*. Here the group serves a second purpose too: correct and incorrect completions of the same prompt become the raw material for preference pairs in Breakout Room #2.

In [5]:
from dataclasses import asdict


@dataclass
class Sample:
    problem: str
    response: str
    extracted: str
    reward: float
    verified_correct: bool


def sample_and_verify(problem: Problem, n_samples: int = 4) -> list[Sample]:
    """Sample a group of completions for one problem and verify each one."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"Solve step by step. Put the final numeric answer in \\boxed{{}}.\n\n{problem.question}",
            system="You are a careful mathematician. Show your work, then box the final number.",
        )
        extracted = extract_number(response)
        reward = reward_fn.compute(response, problem.answer)
        group.append(
            Sample(
                problem=problem.question,
                response=response,
                extracted=extracted,
                reward=reward,
                verified_correct=reward > 0,
            )
        )
    return group


groups = [sample_and_verify(p) for p in problems]

total = sum(len(g) for g in groups)
correct = sum(s.verified_correct for g in groups for s in g)
print(f"Verified-correct rate: {correct / total:.0%} ({correct}/{total})\n")
for problem, group in zip(problems, groups):
    print(f"  {sum(s.verified_correct for s in group)}/{len(group)}  {problem.question[:70]}")

Verified-correct rate: 100% (20/20)

  4/4  What is 12 * 13?
  4/4  If a train covers 60 km in 45 minutes, what is its speed in km/h?
  4/4  What is the sum of the first 10 positive integers?
  4/4  A store discounts a $250 jacket by 20%, then adds 10% sales tax on the
  4/4  How many positive divisors does 360 have?


> NOTE: If your verified-correct rate is 100%, the contrast that drives learning is missing — swap in a smaller model, raise the temperature, or add harder problems until some samples fail.

Let's inspect one failure — reading verifier-rejected completions is how you learn what your policy actually gets wrong (arithmetic slips? misread units? unparseable formatting?):

In [6]:
incorrect = [s for g in groups for s in g if not s.verified_correct]
if incorrect:
    sample = incorrect[0]
    print(f"Problem:   {sample.problem}")
    print(f"Extracted: {sample.extracted!r}  (ground truth mismatch, reward {sample.reward})\n")
    print(sample.response)
else:
    print("No incorrect samples this run - try a harder problem or higher temperature.")

No incorrect samples this run - try a harder problem or higher temperature.


## Breakout Room #1 Summary

- Verifiable rewards come from deterministic checkers, not human preference or a learned reward model — cheap, objective, reproducible.
- The `\boxed{}` convention plus numeric normalization makes extraction mechanical; a brittle verifier punishes correct reasoning and corrupts the signal.
- Asymmetric rewards (+1.0 / −0.1) keep early training from collapsing into refusal.
- Sampling *groups* of completions per prompt is the same structure GRPO trains on — and it produces the correct/incorrect contrast that preference data needs.

---
# Breakout Room #2
## Reward Hacking, Code Verification, and Preference Data

A verifier is not just a metric — once you train against it, it *is* the objective. This room covers what happens then: policies that exploit the verifier's blind spots, a second verifiable domain (code judged by unit tests), and turning audited verifier output into preference data.

## Task 5: Reward-Hacking Detection and the Audit Trail

Goodhart's law — *"when a measure becomes a target, it ceases to be a good measure"* — is the central operational risk of RLVR. A policy optimized against an exact-match verifier will happily learn to:

- emit a bare boxed answer with no reasoning (guessing is cheap when only the box is checked),
- parrot numbers that appear in the prompt,
- or exploit extraction quirks instead of solving the problem.

Two defenses, both borrowed from how production RLVR systems are reviewed:

1. **Flag hack signatures.** Here we flag verified-correct samples that show *no visible work* — a right answer without reasoning is the classic signature of guessing or leakage.
2. **Log every verifier decision** to an append-only audit trail (`artifacts/verifier.jsonl`). When someone — a teammate, an auditor, a regulator — asks whether your RL run was trained on honest rewards, this file is the answer.

In [7]:
import json
from pathlib import Path

AUDIT_LOG = Path("artifacts/verifier.jsonl")
AUDIT_LOG.parent.mkdir(exist_ok=True)


def looks_like_hack(sample: Sample) -> bool:
    """Flag verified-correct samples that show no work.

    A correct boxed answer with no visible reasoning is the classic hack
    signature: the policy may be guessing, pattern-matching the prompt, or
    exploiting the extractor rather than solving the problem.
    """
    if not sample.verified_correct:
        return False
    work = sample.response.replace(f"\\boxed{{{sample.extracted}}}", "")
    numbers_in_work = re.findall(r"-?\d+\.?\d*", work)
    return len(sample.response.split()) < 20 or len(numbers_in_work) < 2


def audit_record(sample: Sample) -> dict:
    """Append one verifier decision to the audit trail and return it."""
    record = {**asdict(sample), "suspected_hack": looks_like_hack(sample)}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    return record


records = [audit_record(s) for g in groups for s in g]
flagged = sum(r["suspected_hack"] for r in records)

print(f"Audited {len(records)} samples -> {flagged} flagged as hack-suspect")
print(f"Audit trail: {AUDIT_LOG} ({sum(1 for _ in AUDIT_LOG.open())} records total)")

Audited 20 samples -> 0 flagged as hack-suspect
Audit trail: artifacts/verifier.jsonl (20 records total)


#### ❓ Question #3

Our detector flags one signature: "right answer, no visible work." Name **two other ways** a policy could hack a `\boxed{}` exact-match verifier, and for each, describe how you would harden the verifier or the prompt against it. (Session 15's stacked format rewards are one relevant hardening example.)

##### Answer:


* **First hack**: parroting or guessing numbers that already appear in the prompt, rather than deriving the answer through reasoning. If a ground-truth number (or something close to it) happens to be present in the problem text, the policy can learn to copy nearby numbers into the box instead of solving anything. Hardening: never let the ground truth or a near-match appear verbatim in the prompt, and add a check that flags boxed answers which are simple copies of numbers already present in the input.

* **Second hack:** gaming the extraction regex itself, for example emitting multiple \boxed{} tags (hoping the checker picks whichever one matches) or padding the boxed value with formatting tricks that survive numeric normalization but reflect no real computation. Hardening: require exactly one \boxed{} per response and penalize or reject multiple boxes, tighten the normalization tolerance so near-matches don't slip through, and add stacked format rewards (as in Session 15) that separately score "did you show a coherent derivation" rather than only checking the final number.

## Task 6: A Code Verifier

Math is one verifiable domain; **code judged by unit tests** is the other workhorse of RLVR. The verifier executes a candidate program against test cases and returns the *fraction that pass* — a graded reward in `[0.0, 1.0]` rather than math's binary match.

We run candidates in a subprocess with a timeout: a program that crashes, hangs, or exits non-zero simply earns no credit for that test case.

> ⚠️ We are executing model-generated code on your machine. For this demo the programs are trivial, but note the design: in production, this verifier runs inside a **sandbox** (container, gVisor, firecracker VM) — never on the host.

In [8]:
import subprocess
import sys


class CodeVerifier:
    """Score generated code by the fraction of test cases it passes."""

    timeout_seconds = 5

    def verify(self, code: str, test_cases: list[dict]) -> float:
        passed = 0
        for tc in test_cases:
            try:
                output = self._run(code, tc.get("input", ""))
                if output.strip() == str(tc["expected"]).strip():
                    passed += 1
            except Exception:
                pass  # crash, timeout, or non-zero exit -> no credit for this case
        return passed / len(test_cases) if test_cases else 0.0

    def _run(self, code: str, input_data: str = "") -> str:
        result = subprocess.run(
            [sys.executable, "-c", code],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=self.timeout_seconds,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout


verifier = CodeVerifier()

# Sanity check with hand-written candidates: one correct, one buggy.
tests = [{"input": "3", "expected": "14"}, {"input": "10", "expected": "385"}]
good = "n = int(input()); print(sum(i * i for i in range(1, n + 1)))"
bad = "n = int(input()); print(sum(range(1, n + 1)))"  # sums i, not i^2

assert verifier.verify(good, tests) == 1.0
assert verifier.verify(bad, tests) == 0.0
print("code verifier OK")

code verifier OK


Now close the loop: have the **policy** write the program, and let the verifier score it — the exact reward signal a coding-RLVR run trains on.

In [9]:
CODING_TASK = (
    "Write a Python program that reads a single integer n from standard input "
    "and prints the sum of the squares of the integers from 1 to n (inclusive). "
    "Print only the number. Reply with only the code - no markdown fences, no explanation."
)

code_tests = [
    {"input": "1", "expected": "1"},
    {"input": "3", "expected": "14"},
    {"input": "10", "expected": "385"},
]


def strip_fences(text: str) -> str:
    """Remove markdown code fences if the policy ignores instructions."""
    return re.sub(r"^```(?:python)?\s*\n|\n?```\s*$", "", text.strip())


for i in range(3):
    candidate = strip_fences(simple_complete(CODING_TASK))
    score = verifier.verify(candidate, code_tests)
    print(f"candidate {i + 1}: reward = {score:.2f}")
    print("  " + candidate.replace("\n", "\n  ") + "\n")

candidate 1: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))

candidate 2: reward = 1.00
  n = int(input())
  print(sum(i**2 for i in range(1, n + 1)))

candidate 3: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))



#### ❓ Question #4

The code verifier returns *fractional* rewards (fraction of tests passed) while the math verifier is binary. What are the benefits and risks of partial credit as a training signal? And concretely: what could a policy-generated program do to a verifier that runs candidates directly on the host, and which parts of that threat does our timeout **not** cover?

##### Answer:

*Fractional rewards give the policy partial credit for partial progress, which is valuable because most non-trivial code isn't binary right/wrong on the way to a solution. It lets the model distinguish "close, minor bug" from "completely wrong," giving a smoother gradient for learning instead of an all-or-nothing signal. The risk is that partial credit can be gamed more easily than a binary check: a policy can learn to hardcode outputs for the visible test cases, passing most of them without implementing real logic, and still collect a high fractional reward without generalizing.*

*Concretely, a policy-generated program running directly on the host could read or exfiltrate files, open network connections, spawn child processes, write to or delete arbitrary paths, or attempt a fork bomb or resource exhaustion attack to disrupt the host. Our timeout only bounds wall-clock runtime. It does nothing to stop filesystem access, network calls, memory exhaustion, or any side effect that completes (or that keeps damaging the system) within that time window. Anything that reads sensitive data, writes to disk, or opens a socket succeeds well before a 5-second timeout would even trigger.*

## Task 7: Build Preference Pairs

Finally, we turn audited verifier output into training data. Within each group:

- **chosen** = verified-correct samples that were *not* flagged as hack-suspect,
- **rejected** = verified-incorrect samples,

and we take the cross product. The resulting `{prompt, chosen, rejected}` records are exactly the format [DPO-style trainers](https://huggingface.co/docs/trl/dpo_trainer) consume — while GRPO (Session 15) skips the pairing and uses the group rewards directly. Same verifier, two consumers.

Excluding flagged samples matters: a hack-suspect completion used as "chosen" would teach the next policy iteration to hack *more*.

In [10]:
def build_preferences(groups: list[list[Sample]], records: list[dict]) -> list[dict]:
    """Cross verified-correct (unflagged) winners with incorrect losers, per group."""
    flagged_responses = {r["response"] for r in records if r["suspected_hack"]}
    pairs = []
    for group in groups:
        winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
        losers = [s for s in group if not s.verified_correct]
        pairs.extend(
            {"prompt": winner.problem, "chosen": winner.response, "rejected": loser.response}
            for winner in winners
            for loser in losers
        )
    return pairs


pairs = build_preferences(groups, records)

PREFERENCES = Path("artifacts/preferences.jsonl")
with PREFERENCES.open("w") as f:
    for pair in pairs:
        f.write(json.dumps(pair) + "\n")

print(f"{len(pairs)} preference pairs -> {PREFERENCES}\n")
if pairs:
    example = pairs[0]
    print(f"prompt:   {example['prompt']}")
    print(f"chosen:   {example['chosen'][:120]}...")
    print(f"rejected: {example['rejected'][:120]}...")
else:
    print("No pairs this run - you need at least one correct AND one incorrect sample in the same group.")

0 preference pairs -> artifacts/preferences.jsonl

No pairs this run - you need at least one correct AND one incorrect sample in the same group.


#### 🏗️ Activity #1: Build Your Own Verifier

Math answers and unit tests are only two verifiable domains. Pick another — for example:

- **JSON schema conformance**: does the completion parse and validate against a schema?
- **SQL correctness**: does a generated query return the same rows as a reference query on a fixture database?
- **Regex/string transformation**: does the output match a deterministic expected transformation of the input?

Then, in the cell below:

1. Implement a reward function for your domain (binary or fractional — justify the choice).
2. Run the sample-and-verify loop over at least 3 prompts with `n_samples >= 3`.
3. Report the verified-correct rate, and note any hack-suspect behavior you observe (and how you'd detect it).

In [12]:
# ============================================================
# Activity #1: A Verifiable Reward for Wardrobe Queries
# ============================================================
# "Is this outfit stylish?" is NOT verifiable (taste, no ground truth).
# "Does this outfit satisfy the hard constraints?" IS verifiable:
# a closed wardrobe catalog + deterministic rules = a program can check it.

import statistics

# ---- 1. The wardrobe: the closed catalog is what makes this checkable ----
WARDROBE = {
    "t1": {"name": "white oxford shirt",     "slot": "top",    "formality": 4, "warmth": 1, "waterproof": False},
    "t2": {"name": "grey wool sweater",      "slot": "top",    "formality": 3, "warmth": 4, "waterproof": False},
    "t3": {"name": "black graphic tee",      "slot": "top",    "formality": 1, "warmth": 1, "waterproof": False},
    "t4": {"name": "navy silk blouse",       "slot": "top",    "formality": 5, "warmth": 1, "waterproof": False},
    "b1": {"name": "charcoal wool trousers", "slot": "bottom", "formality": 4, "warmth": 3, "waterproof": False},
    "b2": {"name": "blue jeans",             "slot": "bottom", "formality": 2, "warmth": 2, "waterproof": False},
    "b3": {"name": "black tailored skirt",   "slot": "bottom", "formality": 5, "warmth": 1, "waterproof": False},
    "b4": {"name": "fleece joggers",         "slot": "bottom", "formality": 1, "warmth": 4, "waterproof": False},
    "s1": {"name": "black leather oxfords",  "slot": "shoes",  "formality": 5, "warmth": 2, "waterproof": False},
    "s2": {"name": "white sneakers",         "slot": "shoes",  "formality": 2, "warmth": 1, "waterproof": False},
    "s3": {"name": "waterproof ankle boots", "slot": "shoes",  "formality": 3, "warmth": 3, "waterproof": True},
    "o1": {"name": "beige trench coat",      "slot": "outer",  "formality": 4, "warmth": 3, "waterproof": True},
    "o2": {"name": "black wool overcoat",    "slot": "outer",  "formality": 5, "warmth": 5, "waterproof": False},
    "o3": {"name": "denim jacket",           "slot": "outer",  "formality": 2, "warmth": 2, "waterproof": False},
    "o4": {"name": "packable rain shell",    "slot": "outer",  "formality": 2, "warmth": 1, "waterproof": True},
}

REQUIRED_SLOTS = {"top", "bottom", "shoes"}


def warmth_band(temp_c: int) -> tuple[int, int]:
    """Acceptable total outfit warmth for a temperature."""
    if temp_c >= 20:
        return (3, 6)
    if temp_c >= 10:
        return (6, 10)
    return (10, 15)


@dataclass
class WardrobeQuery:
    question: str
    min_formality: int
    temp_c: int
    raining: bool


wardrobe_queries = [
    WardrobeQuery("Coffee with a friend at a casual cafe.", 1, 22, False),
    WardrobeQuery("Client dinner at a smart restaurant.", 4, 12, False),
    WardrobeQuery("Walking the dog around the block.", 1, 4, True),
]


def parse_outfit(text: str) -> list[str] | None:
    r"""Extraction, same spirit as extract_number: JSON object first, bare ids as fallback."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            outfit = json.loads(match.group(0)).get("outfit")
            if isinstance(outfit, list):
                return [str(i).strip() for i in outfit]
        except json.JSONDecodeError:
            pass
    ids = re.findall(r"\b[tbso]\d\b", text)
    return ids or None


# ---- 2. The reward function: six independent constraint checks ----
class WardrobeRewardFunction:
    """Fractional verifiable reward: the share of hard constraints an outfit satisfies.

    Fractional, not binary, because the constraints are independent like unit
    tests: an outfit built from real items at the right formality that misses
    the rain rule is genuinely closer to correct than one that hallucinates
    items. Binary would collapse "one rule missed" and "nothing valid" into the
    same score and throw away most of the gradient.
    """

    def checks(self, response: str, query: WardrobeQuery) -> dict[str, bool]:
        outfit = parse_outfit(response)
        results = dict.fromkeys(
            ["parses", "items_exist", "slots_covered", "formality_ok", "warmth_ok", "rain_ok"], False
        )
        if not outfit:
            return results
        results["parses"] = True

        results["items_exist"] = all(i in WARDROBE for i in outfit)
        known = [WARDROBE[i] for i in outfit if i in WARDROBE]
        if not known:
            return results

        slots = [item["slot"] for item in known]
        results["slots_covered"] = REQUIRED_SLOTS.issubset(slots) and all(
            slots.count(s) == 1 for s in slots
        )
        results["formality_ok"] = all(item["formality"] >= query.min_formality for item in known)

        low, high = warmth_band(query.temp_c)
        results["warmth_ok"] = low <= sum(item["warmth"] for item in known) <= high
        results["rain_ok"] = (not query.raining) or any(item["waterproof"] for item in known)
        return results

    def compute(self, response: str, query: WardrobeQuery) -> float:
        results = self.checks(response, query)
        return sum(results.values()) / len(results)


wardrobe_reward_fn = WardrobeRewardFunction()

assert wardrobe_reward_fn.compute('{"outfit": ["t3", "b2", "s2"]}', wardrobe_queries[0]) == 1.0
assert wardrobe_reward_fn.compute('{"outfit": ["t9", "b7", "s5"]}', wardrobe_queries[0]) < 0.5
assert wardrobe_reward_fn.compute("I would wear something nice!", wardrobe_queries[0]) == 0.0
print("wardrobe reward function OK")


# ---- 3. Sample and verify ----
def render_wardrobe() -> str:
    return "\n".join(
        f'{i}: {d["name"]} (slot={d["slot"]}, formality={d["formality"]}/5, '
        f'warmth={d["warmth"]}/5, waterproof={str(d["waterproof"]).lower()})'
        for i, d in WARDROBE.items()
    )


def sample_and_verify_outfit(query: WardrobeQuery, n_samples: int = 3) -> list[Sample]:
    low, high = warmth_band(query.temp_c)
    prompt = (
        f"Wardrobe:\n{render_wardrobe()}\n\n"
        f"Occasion: {query.question}\n"
        f"Minimum formality: {query.min_formality}/5. "
        f"Temperature: {query.temp_c} C, so total outfit warmth must be between {low} and {high}. "
        f"Raining: {str(query.raining).lower()}.\n\n"
        'Reply with only JSON: {"outfit": ["id", ...], "why": "one short sentence"}. '
        "Use at most one item per slot, and always include top, bottom and shoes."
    )
    group = []
    for _ in range(n_samples):
        response = simple_complete(prompt, system="You are a personal stylist. Reply with JSON only.")
        outfit = parse_outfit(response) or []
        reward = wardrobe_reward_fn.compute(response, query)
        group.append(
            Sample(
                problem=query.question,
                response=response,
                extracted=",".join(outfit),
                reward=reward,
                verified_correct=reward == 1.0,  # full credit only when every rule holds
            )
        )
    return group


wardrobe_groups = [sample_and_verify_outfit(q, n_samples=3) for q in wardrobe_queries]

total = sum(len(g) for g in wardrobe_groups)
correct = sum(s.verified_correct for g in wardrobe_groups for s in g)
mean_reward = sum(s.reward for g in wardrobe_groups for s in g) / total

print(f"\nVerified-correct rate: {correct / total:.0%} ({correct}/{total})  |  mean reward: {mean_reward:.2f}\n")

# Within-group spread matters more than the headline rate: a uniform group has
# zero advantage under GRPO and yields no preference pairs under DPO.
for query, group in zip(wardrobe_queries, wardrobe_groups):
    rewards = [s.reward for s in group]
    spread = statistics.pstdev(rewards)
    usable = "usable signal" if spread > 0 else "NO GRADIENT (uniform group)"
    scores = ", ".join(f"{r:.2f}" for r in rewards)
    print(
        f"  {sum(s.verified_correct for s in group)}/{len(group)}  [{scores}]  "
        f"std={spread:.2f}  {usable}  {query.question}"
    )

# Which constraint is actually failing? This is where you learn what the policy gets wrong.
print("\nPer-constraint pass rate:")
for name in wardrobe_reward_fn.checks("", wardrobe_queries[0]):
    hits = sum(
        wardrobe_reward_fn.checks(s.response, q)[name]
        for q, g in zip(wardrobe_queries, wardrobe_groups)
        for s in g
    )
    print(f"  {name:<14} {hits}/{total}")


# ---- 4. Hack detection + audit trail ----
def wardrobe_hack_flags(groups: list[list[Sample]], queries: list[WardrobeQuery]) -> list[dict]:
    """Two domain-specific hack signatures, evaluated only on samples that earned reward.

    Like the notebook's looks_like_hack, these gate on verified_correct: a
    signature only counts as reward hacking if the behaviour is actually
    collecting reward. A repeated wrong answer is just a wrong answer.

    reused_outfit: the same winning item set returned for queries with
        different constraints, meaning the policy found one outfit that scores
        everywhere and stopped reading the query.
    overdressed: a full-credit outfit for a casual occasion built from
        max-formality items. The verifier cannot punish this, because
        formality is checked as a floor (>=), never as a ceiling.
    """
    seen: dict[str, set[str]] = {}
    for query, group in zip(queries, groups):
        for s in group:
            if s.verified_correct and s.extracted:
                seen.setdefault(s.extracted, set()).add(query.question)

    flags = []
    for query, group in zip(queries, groups):
        for s in group:
            if not s.verified_correct:
                flags.append({"reused_outfit": False, "overdressed": False})
                continue
            items = [WARDROBE[i] for i in s.extracted.split(",") if i in WARDROBE]
            mean_formality = sum(i["formality"] for i in items) / len(items) if items else 0
            flags.append(
                {
                    "reused_outfit": len(seen.get(s.extracted, set())) > 1,
                    "overdressed": query.min_formality <= 2 and mean_formality >= 4,
                }
            )
    return flags


wardrobe_flags = wardrobe_hack_flags(wardrobe_groups, wardrobe_queries)
wardrobe_records = []
for sample, flag in zip([s for g in wardrobe_groups for s in g], wardrobe_flags):
    record = {**asdict(sample), "domain": "wardrobe", **flag, "suspected_hack": any(flag.values())}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    wardrobe_records.append(record)

print(
    f"\nAudited {len(wardrobe_records)} samples -> "
    f"{sum(r['reused_outfit'] for r in wardrobe_records)} reused-outfit, "
    f"{sum(r['overdressed'] for r in wardrobe_records)} overdressed, "
    f"{sum(r['suspected_hack'] for r in wardrobe_records)} hack-suspect total"
)


# ---- 5. Close the loop: same Task 7 pattern, new domain ----
flagged_responses = {r["response"] for r in wardrobe_records if r["suspected_hack"]}
wardrobe_pairs = []
for group in wardrobe_groups:
    winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
    losers = [s for s in group if not s.verified_correct]
    wardrobe_pairs.extend(
        {"prompt": w.problem, "chosen": w.response, "rejected": l.response}
        for w in winners
        for l in losers
    )

print(f"\n{len(wardrobe_pairs)} preference pairs from this run")
if not wardrobe_pairs:
    print("  Every group was uniform, so this run produced verified rewards but no training signal.")

wardrobe reward function OK

Verified-correct rate: 33% (3/9)  |  mean reward: 0.87

  0/3  [0.83, 0.83, 0.83]  std=0.00  NO GRADIENT (uniform group)  Coffee with a friend at a casual cafe.
  0/3  [0.83, 0.83, 0.67]  std=0.08  usable signal  Client dinner at a smart restaurant.
  3/3  [1.00, 1.00, 1.00]  std=0.00  NO GRADIENT (uniform group)  Walking the dog around the block.

Per-constraint pass rate:
  parses         9/9
  items_exist    9/9
  slots_covered  9/9
  formality_ok   7/9
  warmth_ok      4/9
  rain_ok        9/9

Audited 9 samples -> 0 reused-outfit, 0 overdressed, 0 hack-suspect total

0 preference pairs from this run
  Every group was uniform, so this run produced verified rewards but no training signal.


### 📝 Activity Notes

**Domain chosen**: *Wardrobe queries. Style is not verifiable, but constraint satisfaction is. A closed wardrobe catalog turns "what should I wear" into six deterministic checks: items exist, slots covered, formality floor, warmth band, rain rule, valid JSON.*

**Binary or fractional**: *Fractional. The six checks are independent like unit tests, so an outfit of real items that only misses the rain rule is genuinely closer to correct than one that invents item ids. I kept verified_correct strict at reward == 1.0, so partial credit shapes the signal but only fully valid outfits become preference-pair winners.*

**Results**: *33% verified-correct (3/9), mean reward 0.87. Parses, items_exist, slots_covered and rain_ok were all 9/9, so no reward was lost to verifier brittleness. Every failure was a reasoning failure, concentrated in one check: warmth, at 4/9.*

**Where the policy fails**: *Not on the hard queries. The cold, rainy dog walk was solved 3/3, while the mild 22 degree cafe failed 3/3, and since that query has a formality floor of 1, all three failures are warmth. Warmth is the only check that needs a sum across the outfit and the only one with an upper bound, which is exactly what the policy keeps crossing.*

**The finding that mattered most**: *33% success produced zero preference pairs. Two of three groups were uniform, so they carry no contrast at all. Verified-correct rate does not tell you a run has training signal. Within-group variance does, which is why I print it.*

**Hack detection, and a bug in my own detector**: *My first version flagged 3 reused outfits, but all three had failed verification. Query-invariance is only hacking when the invariant answer collects reward, so gating flags on verified_correct, as the notebook's own looks_like_hack does, removed all three false positives. An audit trail full of false positives is worse than none, because it teaches you to ignore it.*

**The blind spot I left in on purpose**: *Formality is a floor (>=), so overdressing is never penalized. A silk blouse and leather oxfords to walk the dog would score 1.00. I flag it rather than fix it, because a verifier that visibly misses a wrong answer is the point of this room. The fix is a formality ceiling, which is exactly what warmth already has, and warmth is the check the policy actually fails.*

## Breakout Room #2 Summary

- Once you train against a verifier, it *is* the objective — Goodhart's law makes reward-hacking detection and an append-only audit trail (`artifacts/verifier.jsonl`) part of the core pipeline, not an afterthought.
- Code verification generalizes the idea: unit tests yield fractional rewards, and executing untrusted generated code demands sandboxing in anything beyond a demo.
- Verified groups become training data two ways: `{prompt, chosen, rejected}` pairs for DPO-style trainers, or raw group rewards for GRPO — with hack-suspect samples excluded so the next policy doesn't learn to cheat.

Where to go next: feed `artifacts/preferences.jsonl` to TRL's [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer); plug these reward functions into Session 15's GRPO run; or read how the labs do it at scale — [Tülu 3](https://arxiv.org/abs/2411.15124) (which coined RLVR) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

---
## Conclusion: What We Built, Start to Finish

Walking back through the notebook, the full RLVR arc was:

1. **A policy** (Task 1) — a small API model sampled at temperature 1.0, chosen precisely because it is *sometimes wrong*.
2. **A verifiable domain** (Task 2) — math problems with ground-truth answers and a `\boxed{}` convention that makes checking mechanical.
3. **A reward function** (Task 3) — deterministic, normalized, and asymmetric (+1.0 / −0.1) so early failure doesn't teach refusal.
4. **The sampling loop** (Task 4) — *groups* of completions per prompt, the same structure GRPO computes advantages over, and the source of correct/incorrect contrast.
5. **Adversarial thinking** (Task 5) — once you train against a verifier it *is* the objective, so hack detection and an append-only audit trail are part of the pipeline, not an afterthought.
6. **A second domain** (Task 6) — code judged by unit tests, showing rewards can be fractional and that verification can mean *executing untrusted output*.
7. **Training data** (Task 7) — audited verifier decisions became `{prompt, chosen, rejected}` pairs, with hack-suspects excluded so the next policy iteration doesn't learn to cheat.

The single idea underneath all of it: **wherever a deterministic program can check correctness, you can turn cheap inference into training signal** — no human labelers, no learned reward model. The verifier's quality *is* the ceiling on the policy's quality.

## What Looks Different in Production

This notebook is the smallest honest version of RLVR. Scaling it into a real training pipeline changes nearly every component:

**Sandboxed code execution.** Our `CodeVerifier` runs model-generated code in a bare `subprocess` on your machine — fine for a demo, unacceptable in production, where the policy *will* eventually generate code that reads the filesystem, opens sockets, or fork-bombs the host (and our timeout catches none of that). Production verifiers execute candidates in isolated, disposable environments: [Vercel Sandbox](https://vercel.com/docs/vercel-sandbox) is a good example — ephemeral microVMs built to run untrusted, LLM-generated code with CPU/memory limits, network controls, and full teardown after each run. Self-hosted equivalents include gVisor, Firecracker microVMs, or locked-down containers. The rule: the verifier defines the reward, so the verifier's execution environment is a *security boundary*.

**Scale and throughput.** Five problems × 4 samples becomes tens of thousands of prompts × 8–64 samples per RL step. Sequential API calls give way to async/batched sampling against a dedicated inference fleet (vLLM, as in Session 15) — generation throughput, not the policy update, is usually the bottleneck.

**Verifier hardening.** Regex extraction and exact match get replaced by symbolic math checkers (e.g., SymPy-based equivalence), multiple test suites with hidden held-out cases, and stacked format rewards — because at scale, every blind spot *will* be found and exploited.

**Governance.** Our `verifier.jsonl` becomes real infrastructure: versioned datasets, per-run ledgers, flagged-sample review queues, and dashboards tracking reward distributions for drift. When someone asks "was this model trained on honest rewards?", the audit trail is the answer — this is exactly the artifact regulated environments (the origin of this material) require.

**The training loop itself.** The preference pairs here feed an actual policy update — DPO or GRPO — and then the loop *repeats* against the updated policy: sample, verify, update, again. One pass through this notebook is a single iteration of that flywheel.